# Google TabFM End-to-End Tutorial: Telecom Churn Prediction

This notebook builds a production-style churn prediction workflow for a telecom use case using **Google TabFM** and a strong tree baseline (**XGBoost**).

## Business problem
A telecom company is losing subscribers monthly. Retaining a customer is cheaper than acquiring a new one. We need to predict likely churners in the next 30 days and turn scores into actionable retention policies.

## What you will learn
1. How to download and validate a realistic churn dataset directly from the web.
2. How to train TabFM in multiple modes (default, ensemble preset, advanced custom).
3. How to benchmark against XGBoost with leakage-safe train/val/test handling.
4. How to operationalize predictions using:
   - pure classification metrics,
   - top-k retention targeting,
   - cost-sensitive threshold optimization.

> Note: TabFM model weights are released under a non-commercial license. Use for research/evaluation unless you have appropriate commercial rights.


## 0) Environment setup and reproducibility

We set global seeds and create a deterministic run context. We also prefer GPU if available (your request), with CPU fallback for reliability.


In [ ]:
from __future__ import annotations

import asyncio
import json
import os
import random
import ssl
import urllib.request
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
import torch
from loguru import logger
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

from tabfm import TabFMClassifier
from tabfm import tabfm_v1_0_0_pytorch as tabfm_v1_0_0

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logger.info("Torch version: {}", torch.__version__)
logger.info("Using device: {}", DEVICE)
if DEVICE == "cpu":
    logger.warning("CUDA is unavailable. Running CPU-safe profile for end-to-end execution.")

ARTIFACT_DIR = Path("artifacts")
DATA_DIR = Path("data/raw")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")


STRICT_E2E = os.getenv('STRICT_E2E', '0').strip() == '1'
TABFM_CONTEXT_MAX_ROWS_OVERRIDE = int(os.getenv('TABFM_CONTEXT_MAX_ROWS', '0'))
TABFM_FAST_MODE = os.getenv('TABFM_FAST_MODE', '0').strip() == '1'
logger.info('STRICT_E2E={} TABFM_FAST_MODE={} TABFM_CONTEXT_MAX_ROWS_OVERRIDE={}', STRICT_E2E, TABFM_FAST_MODE, TABFM_CONTEXT_MAX_ROWS_OVERRIDE)

TABFM_CHECKPOINT_OVERRIDE = os.getenv('TABFM_CHECKPOINT_PATH', '').strip()
logger.info('TABFM_CHECKPOINT_OVERRIDE={}', TABFM_CHECKPOINT_OVERRIDE if TABFM_CHECKPOINT_OVERRIDE else None)


## 1) Download telecom churn data directly from the web

We use the public IBM Telco churn dataset (widely used, realistic customer/account/service features).


In [ ]:
TELCO_URLS = [
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv",
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/main/data/Telco-Customer-Churn.csv",
    "https://raw.githubusercontent.com/blastchar/telco-customer-churn/master/WA_Fn-UseC_-Telco-Customer-Churn.csv",
]

EXPECTED_COLUMNS = {
    "customerID",
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "tenure",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "MonthlyCharges",
    "TotalCharges",
    "Churn",
}


def _download_bytes(url: str, timeout: int = 45) -> bytes:
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    context = ssl.create_default_context()
    with urllib.request.urlopen(request, timeout=timeout, context=context) as response:
        return response.read()


async def download_telco_dataset(output_path: Path, urls: list[str], force_refresh: bool = False) -> Path:
    if output_path.exists() and not force_refresh:
        logger.info("Using cached dataset at {}", output_path)
        return output_path

    output_path.parent.mkdir(parents=True, exist_ok=True)
    last_error: Exception | None = None
    for url in urls:
        try:
            logger.info("Attempting download from {}", url)
            payload = await asyncio.to_thread(_download_bytes, url)
            output_path.write_bytes(payload)
            logger.info("Downloaded dataset to {}", output_path)
            return output_path
        except Exception as exc:  # noqa: BLE001
            last_error = exc
            logger.warning("Download failed for {} with error: {}", url, exc)
    raise RuntimeError("All dataset download URLs failed.") from last_error


def validate_schema(df: pd.DataFrame, expected_columns: set[str]) -> None:
    missing = expected_columns - set(df.columns)
    if missing:
        raise ValueError(f"Dataset missing required columns: {sorted(missing)}")


dataset_path = await download_telco_dataset(DATA_DIR / "telco_customer_churn.csv", TELCO_URLS)
raw_df = pd.read_csv(dataset_path)
validate_schema(raw_df, EXPECTED_COLUMNS)
logger.info("Raw shape: {}", raw_df.shape)
raw_df.head()


## 2) Data preparation and leakage-safe splitting

We clean schema issues (`TotalCharges`), encode target, and create **train/val/test** splits.
All preprocessing is fitted only on train for each model pipeline.

### Feature mapping to business problem
The dataset includes core churn drivers close to your requested features: monthly bill (`MonthlyCharges`), contract type (`Contract`), internet plan (`InternetService`), payment behavior proxies (`PaymentMethod`, `PaperlessBilling`, `TotalCharges`), and tenure (`tenure`).


In [ ]:
def prepare_splits(
    raw_data: pd.DataFrame,
    target_col: str = "Churn",
    customer_id_col: str = "customerID",
    seed: int = 42,
    val_size: float = 0.2,
    test_size: float = 0.2,
) -> dict[str, Any]:
    df = raw_data.copy()
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())
    df[target_col] = (df[target_col].astype(str).str.strip().str.lower() == "yes").astype(int)

    drop_cols = [target_col]
    if customer_id_col in df.columns:
        drop_cols.append(customer_id_col)

    X = df.drop(columns=drop_cols)
    y = df[target_col].astype(int).to_numpy()

    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=test_size, random_state=seed, stratify=y
    )
    adjusted_val_size = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val,
        y_train_val,
        test_size=adjusted_val_size,
        random_state=seed,
        stratify=y_train_val,
    )

    return {
        "X_train": X_train.reset_index(drop=True),
        "X_val": X_val.reset_index(drop=True),
        "X_test": X_test.reset_index(drop=True),
        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test,
        "full_clean": df.reset_index(drop=True),
    }


splits = prepare_splits(raw_df, seed=SEED)
for key in ["X_train", "X_val", "X_test"]:
    logger.info("{} shape: {}", key, splits[key].shape)

train_rate = splits["y_train"].mean()
val_rate = splits["y_val"].mean()
test_rate = splits["y_test"].mean()
logger.info("Churn rate train/val/test: {:.3f} / {:.3f} / {:.3f}", train_rate, val_rate, test_rate)

splits["X_train"].head()


## 3) Evaluation helpers

Centralized metric functions keep model comparisons consistent and production-ready.


In [ ]:
def evaluate_classifier(
    y_true: np.ndarray,
    y_score: np.ndarray,
    threshold: float = 0.5,
) -> dict[str, float]:
    y_pred = (y_score >= threshold).astype(int)
    return {
        "roc_auc": float(roc_auc_score(y_true, y_score)),
        "pr_auc": float(average_precision_score(y_true, y_score)),
        "brier": float(brier_score_loss(y_true, y_score)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    }


def simulate_top_k_campaign(
    y_true: np.ndarray,
    y_score: np.ndarray,
    k_values: tuple[float, ...] = (0.05, 0.10, 0.20),
    offer_cost: float = 25.0,
    retention_success_prob: float = 0.35,
    saved_customer_value: float = 300.0,
) -> pd.DataFrame:
    n = len(y_true)
    order = np.argsort(-y_score)
    y_sorted = y_true[order]
    rows: list[dict[str, float]] = []
    total_churners = int(y_true.sum())
    for k in k_values:
        target_n = max(1, int(round(n * k)))
        targeted = y_sorted[:target_n]
        tp = int(targeted.sum())
        precision_at_k = tp / target_n
        recall_at_k = tp / max(1, total_churners)
        expected_saved = tp * retention_success_prob
        expected_gross_value = expected_saved * saved_customer_value
        total_cost = target_n * offer_cost
        expected_net_value = expected_gross_value - total_cost
        rows.append(
            {
                "k_fraction": k,
                "target_customers": target_n,
                "true_churners_targeted": tp,
                "precision_at_k": precision_at_k,
                "recall_at_k": recall_at_k,
                "expected_saved_customers": expected_saved,
                "expected_campaign_cost": total_cost,
                "expected_net_value": expected_net_value,
            }
        )
    return pd.DataFrame(rows).sort_values("k_fraction").reset_index(drop=True)


def optimize_cost_sensitive_threshold(
    y_true: np.ndarray,
    y_score: np.ndarray,
    offer_cost: float = 25.0,
    retention_success_prob: float = 0.35,
    saved_customer_value: float = 300.0,
    grid_size: int = 201,
) -> tuple[pd.DataFrame, dict[str, float]]:
    thresholds = np.linspace(0.0, 1.0, grid_size)
    rows: list[dict[str, float]] = []
    for threshold in thresholds:
        targeted = (y_score >= threshold).astype(int)
        targeted_count = int(targeted.sum())
        tp = int(((targeted == 1) & (y_true == 1)).sum())
        expected_saved = tp * retention_success_prob
        expected_gross = expected_saved * saved_customer_value
        expected_cost = targeted_count * offer_cost
        expected_net = expected_gross - expected_cost
        rows.append(
            {
                "threshold": float(threshold),
                "targeted_customers": float(targeted_count),
                "true_churners_targeted": float(tp),
                "expected_saved_customers": float(expected_saved),
                "expected_gross_value": float(expected_gross),
                "expected_campaign_cost": float(expected_cost),
                "expected_net_value": float(expected_net),
            }
        )
    curve_df = pd.DataFrame(rows)
    best_idx = curve_df["expected_net_value"].idxmax()
    best_row = curve_df.loc[best_idx].to_dict()
    return curve_df, best_row


## 4) Baseline model: XGBoost

A strong baseline keeps the project honest: we evaluate TabFM improvements relative to a high-quality tabular baseline.


In [ ]:
def train_xgboost_baseline(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    seed: int = 42,
) -> Pipeline:
    categorical_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numeric_cols = [col for col in X_train.columns if col not in categorical_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_cols,
            ),
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical_cols,
            ),
        ]
    )

    xgb = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=200,
        learning_rate=0.03,
        max_depth=6,
        min_child_weight=2.0,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        random_state=seed,
        n_jobs=-1,
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", xgb),
        ]
    )
    pipeline.fit(X_train, y_train)
    return pipeline


xgb_model = train_xgboost_baseline(splits["X_train"], splits["y_train"], seed=SEED)
logger.info("Trained XGBoost baseline.")


## 5) TabFM model variants

We train three TabFM variants to cover the model's core capabilities:
- **Default**: straightforward TabFMClassifier setup.
- **Ensemble preset**: TabFM-recommended stronger config.
- **Advanced custom**: explicit control over ensemble generation, feature crosses/SVD, calibration, and NNLS blending.


In [ ]:
def train_tabfm_variants(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    device: str,
    seed: int = 42,
) -> tuple[dict[str, TabFMClassifier], dict[str, Any]]:
    requested_device = device
    effective_device = device

    if device.startswith("cuda") and torch.cuda.is_available():
        gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        logger.info("Detected GPU memory: {:.2f} GiB", gpu_mem_gb)
        if gpu_mem_gb < 12.0:
            effective_device = "cpu"
            logger.warning(
                "GPU memory {:.2f} GiB is too small for stable TabFM ensemble OOF in this notebook. Falling back to CPU-safe compact profile.",
                gpu_mem_gb,
            )

    if STRICT_E2E and not TABFM_FAST_MODE:
        if effective_device.startswith("cuda"):
            profile = {
                "default_estimators": 20,
                "ensemble_estimators": 28,
                "advanced_estimators": 28,
                "batch_size": 2,
                "advanced_norm_methods": ["none", "power", "quantile_rtdl"],
                "max_train_rows": None,
            }
        else:
            profile = {
                "default_estimators": 2,
                "ensemble_estimators": 2,
                "advanced_estimators": 2,
                "batch_size": 1,
                "advanced_norm_methods": ["none", "power"],
                "max_train_rows": None,
            }
    elif effective_device.startswith("cuda"):
        profile = {
            "default_estimators": 16,
            "ensemble_estimators": 24,
            "advanced_estimators": 24,
            "batch_size": 2,
            "advanced_norm_methods": ["none", "power", "quantile_rtdl"],
            "max_train_rows": 1200,
        }
    else:
        profile = {
            "default_estimators": 2,
            "ensemble_estimators": 2,
            "advanced_estimators": 2,
            "batch_size": 1,
            "advanced_norm_methods": ["none"],
            "max_train_rows": 256,
        }

    if TABFM_CONTEXT_MAX_ROWS_OVERRIDE > 0:
        profile['max_train_rows'] = TABFM_CONTEXT_MAX_ROWS_OVERRIDE

    X_fit = X_train
    y_fit = y_train
    if profile["max_train_rows"] is not None and len(X_train) > profile["max_train_rows"]:
        X_fit, _, y_fit, _ = train_test_split(
            X_train,
            y_train,
            train_size=profile["max_train_rows"],
            random_state=seed,
            stratify=y_train,
        )
        logger.warning(
            "TabFM compact-fit mode: using stratified {} rows out of {} training rows.",
            len(X_fit),
            len(X_train),
        )

    logger.info(
        "TabFM profile (requested_device={}, effective_device={}): {}",
        requested_device,
        effective_device,
        json.dumps(profile, indent=2),
    )

    checkpoint_path = TABFM_CHECKPOINT_OVERRIDE if TABFM_CHECKPOINT_OVERRIDE else None
    base_model = tabfm_v1_0_0.load(
        model_type="classification",
        checkpoint_path=checkpoint_path,
        device=effective_device,
    )
    models: dict[str, TabFMClassifier] = {}

    default_clf = TabFMClassifier(
        model=base_model,
        n_estimators=profile["default_estimators"],
        random_state=seed,
        batch_size=profile["batch_size"],
        verbose=False,
    )
    default_clf.fit(X_fit, y_fit)
    models["tabfm_default"] = default_clf

    ensemble_clf = TabFMClassifier.ensemble(
        model=base_model,
        n_estimators=profile["ensemble_estimators"],
        random_state=seed,
        batch_size=profile["batch_size"],
        num_folds_for_cv=2,
        min_rows_for_single_val_split=64,
        enable_nnls=False,
        binary_calibration_method=None,
        multiclass_calibration_method=None,
        n_feature_crosses=0,
        n_svd_features=0,
        verbose=False,
    )
    ensemble_clf.fit(X_fit, y_fit)
    models["tabfm_ensemble_preset"] = ensemble_clf

    advanced_clf = TabFMClassifier(
        model=base_model,
        n_estimators=profile["advanced_estimators"],
        norm_methods=profile["advanced_norm_methods"],
        feat_shuffle_method="random",
        class_shift=True,
        permute_categorical=True,
        n_feature_crosses="sqrt",
        n_svd_features="sqrt",
        total_svd_pool=None,
        average_logits=False,
        enable_nnls=True,
        nnls_beta=0.75,
        binary_calibration_method="platt",
        softmax_temperature=0.9,
        random_state=seed,
        batch_size=profile["batch_size"],
        num_folds_for_cv=2,
        min_rows_for_single_val_split=64,
        verbose=False,
    )
    advanced_clf.fit(X_fit, y_fit)
    models["tabfm_advanced_custom"] = advanced_clf

    run_meta = {
        'requested_device': requested_device,
        'effective_device': effective_device,
        'profile': profile,
        'train_rows_original': int(len(X_train)),
        'train_rows_effective': int(len(X_fit)),
    }
    return models, run_meta


tabfm_models, tabfm_training_meta = train_tabfm_variants(
    splits["X_train"],
    splits["y_train"],
    device=DEVICE,
    seed=SEED,
)
logger.info("Trained TabFM variants: {}", list(tabfm_models.keys()))


## 6) Unified benchmark table (validation + test)

We collect comparable metrics for all models and choose a champion by validation PR-AUC.


In [ ]:
def _collect_scores(model: Any, X: pd.DataFrame) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        return np.asarray(proba)[:, 1].astype(float)
    raise ValueError("Model does not expose predict_proba.")


model_registry: dict[str, Any] = {
    "xgboost_baseline": xgb_model,
    **tabfm_models,
}

rows: list[dict[str, Any]] = []
predictions: dict[str, dict[str, np.ndarray]] = {}

for model_name, model in model_registry.items():
    val_scores = _collect_scores(model, splits["X_val"])
    test_scores = _collect_scores(model, splits["X_test"])
    predictions[model_name] = {"val": val_scores, "test": test_scores}

    val_metrics = evaluate_classifier(splits["y_val"], val_scores, threshold=0.5)
    test_metrics = evaluate_classifier(splits["y_test"], test_scores, threshold=0.5)
    rows.append({"model": model_name, "split": "val", **val_metrics})
    rows.append({"model": model_name, "split": "test", **test_metrics})

metrics_df = pd.DataFrame(rows).sort_values(["split", "pr_auc"], ascending=[True, False]).reset_index(drop=True)
metrics_df


In [ ]:
val_rank = metrics_df[metrics_df["split"] == "val"].sort_values("pr_auc", ascending=False).reset_index(drop=True)
champion_model_name = val_rank.loc[0, "model"]
logger.info("Champion model by validation PR-AUC: {}", champion_model_name)
val_rank


## 7) Diagnostics: ranking, confusion, and calibration

This section addresses the **pure classification** requirement.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
test_plot_df = metrics_df[metrics_df["split"] == "test"].sort_values("pr_auc", ascending=False)
sns.barplot(data=test_plot_df, x="pr_auc", y="model", ax=axes[0], palette="viridis")
axes[0].set_title("Test PR-AUC by model")
axes[0].set_xlabel("PR-AUC")
axes[0].set_ylabel("")

sns.barplot(data=test_plot_df, x="roc_auc", y="model", ax=axes[1], palette="mako")
axes[1].set_title("Test ROC-AUC by model")
axes[1].set_xlabel("ROC-AUC")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()


In [ ]:
champion_scores_test = predictions[champion_model_name]["test"]
champion_pred_test = (champion_scores_test >= 0.5).astype(int)
cm = confusion_matrix(splits["y_test"], champion_pred_test)
cm_df = pd.DataFrame(cm, index=["Actual_NoChurn", "Actual_Churn"], columns=["Pred_NoChurn", "Pred_Churn"])
cm_df


In [ ]:
prob_true, prob_pred = calibration_curve(splits["y_test"], champion_scores_test, n_bins=10, strategy="quantile")

plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfectly calibrated")
plt.plot(prob_pred, prob_true, marker="o", linewidth=2, label=champion_model_name)
plt.title("Calibration curve (champion, test set)")
plt.xlabel("Predicted churn probability")
plt.ylabel("Observed churn frequency")
plt.legend()
plt.tight_layout()
plt.show()


## 8) Retention policy A: Top-k campaign

This section addresses your **top-k targeting** requirement.
The retention team contacts only the highest-risk fraction each month.


In [ ]:
BUSINESS_ASSUMPTIONS = {
    "offer_cost": 25.0,
    "retention_success_prob": 0.35,
    "saved_customer_value": 300.0,
}

topk_df = simulate_top_k_campaign(
    y_true=splits["y_test"],
    y_score=champion_scores_test,
    k_values=(0.05, 0.10, 0.15, 0.20, 0.30),
    **BUSINESS_ASSUMPTIONS,
)
topk_df


## 9) Retention policy B: Cost-sensitive threshold optimization

This section addresses your **cost-sensitive policy** requirement.
We learn the best threshold on validation, then evaluate on test.


In [ ]:
threshold_curve_df, best_threshold_row = optimize_cost_sensitive_threshold(
    y_true=splits["y_val"],
    y_score=predictions[champion_model_name]["val"],
    **BUSINESS_ASSUMPTIONS,
)

best_threshold = float(best_threshold_row["threshold"])
logger.info("Best threshold from validation economics: {:.3f}", best_threshold)
pd.DataFrame([best_threshold_row])


In [ ]:
plt.figure(figsize=(8, 4))
sns.lineplot(data=threshold_curve_df, x="threshold", y="expected_net_value", linewidth=2)
plt.axvline(best_threshold, color="red", linestyle="--", label=f"Best threshold={best_threshold:.3f}")
plt.title("Validation expected net value vs threshold")
plt.xlabel("Decision threshold")
plt.ylabel("Expected net value")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
def evaluate_threshold_business(
    y_true: np.ndarray,
    y_score: np.ndarray,
    threshold: float,
    offer_cost: float,
    retention_success_prob: float,
    saved_customer_value: float,
) -> dict[str, float]:
    targeted = (y_score >= threshold).astype(int)
    targeted_count = int(targeted.sum())
    tp = int(((targeted == 1) & (y_true == 1)).sum())
    expected_saved = tp * retention_success_prob
    expected_gross = expected_saved * saved_customer_value
    expected_cost = targeted_count * offer_cost
    expected_net = expected_gross - expected_cost
    return {
        "threshold": float(threshold),
        "targeted_customers": float(targeted_count),
        "true_churners_targeted": float(tp),
        "expected_saved_customers": float(expected_saved),
        "expected_gross_value": float(expected_gross),
        "expected_campaign_cost": float(expected_cost),
        "expected_net_value": float(expected_net),
    }


test_business = evaluate_threshold_business(
    y_true=splits["y_test"],
    y_score=champion_scores_test,
    threshold=best_threshold,
    **BUSINESS_ASSUMPTIONS,
)
pd.DataFrame([test_business])


## 10) Persist production artifacts

Artifacts include benchmark metrics, prediction tables, and policy outputs for downstream retention ops.


In [ ]:
pred_frame = pd.DataFrame(
    {
        "y_true": splits["y_test"],
        **{f"score_{name}": preds["test"] for name, preds in predictions.items()},
    }
)

metrics_path = ARTIFACT_DIR / "telco_churn_model_metrics.csv"
pred_path = ARTIFACT_DIR / "telco_churn_predictions_test.parquet"
topk_path = ARTIFACT_DIR / "telco_churn_topk_campaign.csv"
threshold_curve_path = ARTIFACT_DIR / "telco_churn_threshold_curve.csv"
threshold_summary_path = ARTIFACT_DIR / "telco_churn_threshold_summary.csv"

metrics_df.to_csv(metrics_path, index=False)
pl.from_pandas(pred_frame).write_parquet(pred_path)
topk_df.to_csv(topk_path, index=False)
threshold_curve_df.to_csv(threshold_curve_path, index=False)
pd.DataFrame([best_threshold_row, test_business]).to_csv(threshold_summary_path, index=False)

logger.info("Saved metrics to {}", metrics_path)
logger.info("Saved test predictions to {}", pred_path)
logger.info("Saved top-k campaign table to {}", topk_path)
logger.info("Saved threshold curve to {}", threshold_curve_path)
logger.info("Saved threshold summary to {}", threshold_summary_path)


runtime_meta_path = ARTIFACT_DIR / "telco_churn_runtime_meta.json"
runtime_meta = {
    "seed": SEED,
    "strict_e2e": bool(STRICT_E2E),
    "tabfm_fast_mode": bool(TABFM_FAST_MODE),
    "tabfm_context_max_rows_override": int(TABFM_CONTEXT_MAX_ROWS_OVERRIDE),
    "tabfm_device_requested": tabfm_training_meta.get("requested_device"),
    "tabfm_device_effective": tabfm_training_meta.get("effective_device"),
    "tabfm_profile": tabfm_training_meta.get("profile", {}),
    "tabfm_train_rows_original": tabfm_training_meta.get("train_rows_original"),
    "tabfm_train_rows_effective": tabfm_training_meta.get("train_rows_effective"),
    "champion_model": champion_model_name,
}
runtime_meta_path.write_text(json.dumps(runtime_meta, indent=2))
logger.info("Saved runtime meta to {}", runtime_meta_path)


## 11) Interpretation and production notes

### What we built
- A full churn risk modeling workflow with robust data ingestion, train/val/test rigor, and benchmarked models.
- Advanced TabFM configurations beyond default mode.
- Action policies for three levels of decisioning: classification, top-k, and cost-sensitive optimization.

### Key assumptions
- `retention_success_prob=0.35`, `offer_cost=25`, `saved_customer_value=300`.
- These should be replaced with business-validated finance/ops estimates before production deployment.

### Next production upgrades
1. Backtest on multiple monthly cohorts and measure policy stability over time.
2. Add drift monitoring on feature and score distributions.
3. Integrate intervention outcomes (accepted offer / retained) to learn causal uplift over time.
